# Enclave Inference — Gemma 3 — Clear State

Wipes the SyftBox of the Model Owner, Benchmark Owner and Researcher so the gemma notebooks can be re-run from
scratch. Same two calls each notebook already has inline:

```python
client._manager.delete_syftbox()
client._manager.peer_manager.write_own_version()
```

Each party is logged in by email, so this needs their cached tokens on this machine. Anyone without a token is
skipped with a warning rather than failing the cell.

## Setup

In [ ]:
!uv pip install -Uq "git+https://github.com/OpenMined/PySyft.git@dev#subdirectory=packages/syft-enclave"

In [ ]:
import os

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds

In [ ]:
MODEL_OWNER_EMAIL     = "test.model.owner@gmail.com"
BENCHMARK_OWNER_EMAIL = "test.benchmark.owner@gmail.com"
RESEARCHER_EMAIL      = "test.researcher@gmail.com"

PARTIES = [
    ("Model owner",     MODEL_OWNER_EMAIL,     login_do),
    ("Benchmark owner", BENCHMARK_OWNER_EMAIL, login_do),
    ("Researcher",      RESEARCHER_EMAIL,      login_ds),
]

print("\n".join(f"  {role:16}: {email}" for role, email, _ in PARTIES))

## Clear

`sync=False, load_peers=False` — no point syncing state we are about to delete.

In [ ]:
for role, email, login in PARTIES:
    try:
        client = login(email=email, sync=False, load_peers=False)
        client._manager.delete_syftbox()
        client._manager.peer_manager.write_own_version()
        print(f"  {role:16}: cleared  ({email})")
    except Exception as e:
        print(f"  {role:16}: SKIPPED  ({email}) — {type(e).__name__}: {e}")

## Clear the enclave too (optional)

Run only if you also want the enclave's own state gone — it drops every dataset and job it holds.

In [ ]:
ENCLAVE_EMAIL = "test.enclave@gmail.com"

enclave = login_do(email=ENCLAVE_EMAIL, sync=False, load_peers=False)
enclave._manager.delete_syftbox()
enclave._manager.peer_manager.write_own_version()
print(f"  Enclave         : cleared  ({ENCLAVE_EMAIL})")